# 1. Get and build an area

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/01_area_to_network.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/01_area_to_network.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

<div style="background:linear-gradient(135deg,#1f6feb,#0b4f8a);color:#fff;border-radius:10px;padding:16px 20px;margin:12px 0"><div style="font-size:0.8em;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85">duckOSM notebook manual · chapter 1</div><div style="font-size:1.15em;margin-top:6px;line-height:1.5">Get the road network of an area into one file: from a place, an OSM id, a box, a point, an H3 cell, a polygon of your own, a map, or an OSM file that you already have.</div></div>

<div style="border:1px solid rgba(128,128,128,0.35);border-radius:8px;padding:8px 16px;margin:10px 0"><table style="border-collapse:collapse;width:100%;margin:0"><tr><td style="width:7em;padding:6px 14px 6px 0;font-weight:700;vertical-align:top;white-space:nowrap">You learn</td><td style="padding:6px 0">the parts of a duckOSM build; seven ways to choose an area; the modes; your own PBF file; the build options and <code>config=</code>; where the files go</td></tr><tr><td style="width:7em;padding:6px 14px 6px 0;font-weight:700;vertical-align:top;white-space:nowrap">Time</td><td style="padding:6px 0">about 3 minutes (the first download included)</td></tr><tr><td style="width:7em;padding:6px 14px 6px 0;font-weight:700;vertical-align:top;white-space:nowrap">Extras</td><td style="padding:6px 0"><code>duckosm[widget,viz]</code> (the install cell installs them)</td></tr><tr><td style="width:7em;padding:6px 14px 6px 0;font-weight:700;vertical-align:top;white-space:nowrap">Next</td><td style="padding:6px 0"><a href="02_inspect.md">2. Look inside the file</a>: the tables, the columns and SQL</td></tr></table></div>

In [1]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
# Until duckOSM 0.9.0 is on PyPI, the SDK of these notebooks is in the nightly wheel of main (the GitHub pre-release "nightly").
import importlib.util, json, urllib.request
def nightly_wheel():       # the address of the newest nightly wheel
    release = json.load(urllib.request.urlopen("https://api.github.com/repos/Khoshkhah/duckOSM/releases/tags/nightly"))
    return next(a["browser_download_url"] for a in release["assets"] if a["name"].endswith(".whl"))
if importlib.util.find_spec("duckosm") is None:      # not installed: install it with the extras of this notebook
    %pip install -q "duckosm[widget,viz] @ {nightly_wheel()}"
import duckosm             # the SDK
if not hasattr(duckosm, "network"):                    # an old duckOSM (0.8.0 or older), without the SDK
    %pip install -q --force-reinstall --no-deps "duckosm @ {nightly_wheel()}"
    raise SystemExit("duckOSM is updated: restart the session (in Colab: Runtime > Restart session), then run this cell again")
print("duckosm", duckosm.__version__)    # the version

duckosm 0.8.0


## The parts of a build

A duckOSM build has three steps. Each step has its own call, and `duckosm.network` does all three in one call.

| Step | What happens | The call | The result |
|---|---|---|---|
| 1. The boundary | The area becomes a polygon: from a name or an OSM id (Nominatim), a box, a point, an H3 cell or your own GeoJSON | part of `duckosm.get` | `<name>.geojson` |
| 2. The OSM data | duckOSM finds the smallest Geofabrik extract that holds the boundary, downloads it once into a cache, and cuts it to the boundary | `duckosm.get` | `<name>.osm.pbf`, and an `Area` |
| 3. The network | The Rust engine reads the cut file and writes the roads of each mode, the nodes, the graph of turns, the base map and more | `area.build` or `duckosm.build` | `<name>.duckdb`, and a `Network` |

The result is **one DuckDB file**. Each mode is a schema in it (`driving`, `walking`, `cycling`), with the tables
`edges` and `nodes`. Each edge has an `edge_id`, a number that stays the same when you build the area again, so your
own data can join to it (chapter 5).

<div style="border-left:4px solid #2ea043;background:rgba(46,160,67,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#2ea043;margin-bottom:4px">Tip</div>You can open the file with any DuckDB tool (the <code>duckdb</code> program, DBeaver, Python's <code>duckdb</code>), not only with duckOSM.</div>

## Quick start: one call

`duckosm.network` gets the area and builds it. Here the area is the city of Monaco, OSM relation 2220322. The
argument `modes` chooses the networks: driving and walking.

In [2]:
net = duckosm.network(osm_id=2220322, modes=["driving", "walking"], name="monaco")    # get and build Monaco
net                                                                                 # the card of the network

cut: 34461 nodes, 5685 ways, 339 relations

Output: data/monaco.duckdb

mode,edges,nodes,km,oneway_km
driving,2840,1768,96.2,35.5
walking,11938,4589,260.6,0.0


<div style="border-left:4px solid #388bfd;background:rgba(56,139,253,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#388bfd;margin-bottom:4px">What the output shows</div>The progress lines name each step: the boundary from Nominatim, the extract and the cut, then the build. The card at the end shows the file and, for each mode, the number of edges and nodes and the length in km. A <code>Network</code> is also the path of its file, so <code>str(net)</code> gives <code>data/monaco.duckdb</code>.</div>

<div style="border-left:4px solid #d29922;background:rgba(210,153,34,0.14);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#d29922;margin-bottom:4px">Warning</div>The name "Monaco" finds the <i>country</i> of Monaco, whose boundary includes a large part of the sea. That is why this chapter uses the OSM id of the city. When a name gives the wrong area, find its OSM id on <a href="https://www.openstreetmap.org">openstreetmap.org</a> (search, then the "Relation" number) and use <code>osm_id=</code>.</div>

## The summary of each mode

`net.summary()` gives one row for each mode: the edges, the nodes, the length in km and the length of the one-way
roads. `net.modes` lists the modes of the file.

In [3]:
print(net.modes)       # the modes in the file
net.summary()          # one row for each mode

['driving', 'walking']


,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5
1,walking,11938,4589,260.6,0.0


<div style="border-left:4px solid #388bfd;background:rgba(56,139,253,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#388bfd;margin-bottom:4px">What the output shows</div>The walking network is much larger than the driving network: it has the footways, the steps and the paths, and both sides of the streets where OSM draws the sidewalks. A two-way road is two edges, one for each direction.</div>

## The modes

A mode is one network. Each mode keeps the roads that it can use, with its own rules.

| Mode | Keeps | Leaves out |
|---|---|---|
| `driving` | the roads for cars (motorway to residential, service roads, ramps), with one-way rules and turn restrictions | footways, paths, steps, private roads |
| `walking` | footways, paths, steps, pedestrian streets, sidewalks, and the roads where people can walk | motorways and roads where walking is not allowed |
| `cycling` | cycleways, roads for bicycles, and footways where you can push the bicycle | motorways, steps |

`net.walking` (or `net["walking"]`) is the view of one mode: the same calls, without `mode=`.

In [4]:
net.walking            # the card of one mode

mode,edges,nodes,km,oneway_km
walking,11938,4589,260.6,0.0


## Seven ways to choose an area

`duckosm.get` does steps 1 and 2 and gives an `Area`: the cut OSM file and its boundary. It does not build. Use it
to look at an area before the build, or to build it later with other options. Each way below is one argument.

| Argument | The area | Example |
|---|---|---|
| `place` | the first answer of Nominatim for a name | `duckosm.get("Monaco")` |
| `osm_id` | an OSM relation (a city, a district, a park) | `duckosm.get(osm_id=2220322)` |
| `bbox` | a box: west, south, east, north, in degrees | `duckosm.get(bbox=(7.41, 43.72, 7.44, 43.75))` |
| `point` and `dist_m` | the box `dist_m` metres around a point (lon, lat) | `duckosm.get(point=(7.4246, 43.7384), dist_m=500)` |
| `h3_cell` | one H3 cell | `duckosm.get(h3_cell="883969a40bfffff")` |
| `boundary` | your own polygon: a GeoJSON file, a GeoJSON dict, or a shapely or GeoPandas object | `duckosm.get(boundary="district.geojson")` |
| `pbf` (in `network`) | an OSM file that you have | see "Your own PBF file" below |

The extract stays in the cache, so each next area in Monaco needs no download.

### A box around a point

`point` with `dist_m` is the same as OSMnx's `graph_from_point`: a box of 2 × 500 m around the casino square.

In [5]:
casino = duckosm.get(point=(7.4280, 43.7393), dist_m=500, name="casino", out="data")   # an Area: the file and the boundary
casino                                                                                  # the card of the area

cut: 11894 nodes, 1804 ways, 181 relations

,file,size
OSM file,data/casino.osm.pbf,0.3 MB
boundary,data/casino.geojson,0 kB


<div style="border-left:4px solid #388bfd;background:rgba(56,139,253,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#388bfd;margin-bottom:4px">What the output shows</div>The progress line counts the nodes, ways and relations of the cut. The card names the two files: the cut OSM file and the boundary. The box is a small part of the extract.</div>

### A box in degrees

`bbox` takes west, south, east and north, in this order (the order of GeoJSON and of OSMnx 2).

In [6]:
harbour = duckosm.get(bbox=(7.418, 43.730, 7.430, 43.738), name="harbour", out="data")   # the harbour of Monaco
harbour

cut: 9883 nodes, 1713 ways, 170 relations

,file,size
OSM file,data/harbour.osm.pbf,0.2 MB
boundary,data/harbour.geojson,0 kB


### Your own polygon

`boundary` takes a polygon. Here it is a shapely polygon; a GeoJSON file or a GeoPandas object work the same way.
duckOSM writes it to `<out>/<name>.geojson`.

In [7]:
from shapely.geometry import Polygon                       # any object with __geo_interface__ works
monte_carlo = Polygon([(7.420, 43.737), (7.432, 43.737), (7.432, 43.745), (7.420, 43.745)])
mc = duckosm.get(boundary=monte_carlo, name="monte_carlo", out="data")
mc

cut: 17612 nodes, 2387 ways, 208 relations

,file,size
OSM file,data/monte_carlo.osm.pbf,0.3 MB
boundary,data/monte_carlo.geojson,0 kB


### Look before you get: `dry_run`

`dry_run=True` downloads and writes nothing. It tells you which extract the area needs, its size, and whether it is
in the cache already. Use it before a large area.

In [8]:
plan = duckosm.get(osm_id=2220322, dry_run=True)        # no download, no file
{k: plan["extract"][k] for k in ("name", "km2", "bytes", "cached")}

boundary: Monaco, OSM relation 2220322, 2.39 km², from Nominatim

{'name': 'Monaco', 'km2': 184.34333987387117, 'bytes': 708128, 'cached': True}

## Build an area

`area.build` does step 3 for an `Area`. Its arguments are the same as those of `duckosm.network`: `modes`, `output`,
`fixes` and the build options (below).

In [9]:
casino_net = casino.build(modes=["walking"])      # the walking network of the box around the casino
casino_net.summary()

Output: data/casino.duckdb

,mode,edges,nodes,km,oneway_km
0,walking,3366,1272,87.3,0.0


## Choose the area on a map

`duckosm.AreaPicker` is a map in the notebook. Choose an area with one of the tabs (Search, OSM id, Box, H3 cell,
File); the line under the map shows the extract and its size. Click **Get**, or call `picker.get()`. You can also set
the area from Python with `picker.area`: the widget then shows its tab and its value.

![The AreaPicker widget: the OSM id 2220322 gives the city of Monaco](https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/images/area-picker.png)

In [10]:
picker = duckosm.AreaPicker(out="data")                 # the files go into the folder data/
picker.area = {"mode": "osm_id", "osm_id": 2220322}      # the area when you choose none on the map
picker                                                    # the map

In [11]:
area = picker.result or picker.get()    # the result of Get, or get it now
area

boundary: Monaco, OSM relation 2220322, 2.39 km², from Nominatim

,file,size
OSM file,data/monaco.osm.pbf,0.6 MB
boundary,data/monaco.geojson,16 kB


## Your own PBF file

`pbf=` takes an `.osm.pbf` file that you already have: from Geofabrik, from BBBike, from your own OSM editor, or a
file that you cut with osmium. Without `boundary=`, the whole file is the area. With it, duckOSM cuts the file to
the boundary before the build. Here the file is a Monaco extract (0.6 MB).

In [12]:
import urllib.request                                                           # your own file: here, a download
mine = "data/my_monaco.osm.pbf"
urllib.request.urlretrieve("https://github.com/Khoshkhah/duckOSM/raw/main/tests/data/monaco.osm.pbf", mine)
whole = duckosm.network(pbf=mine, name="monaco_whole", modes=["driving"])        # the whole file
whole.summary()

Output: data/monaco_whole.duckdb

,mode,edges,nodes,km,oneway_km
0,driving,2844,1769,96.2,35.6


In [13]:
cut = duckosm.network(pbf=mine, boundary="data/monte_carlo.geojson", name="monte_carlo_cut", modes=["driving"])
cut.summary()                                                                     # only the roads in Monte Carlo

Output: data/monte_carlo_cut.duckdb

,mode,edges,nodes,km,oneway_km
0,driving,779,484,28.4,8.9


<div style="border-left:4px solid #388bfd;background:rgba(56,139,253,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#388bfd;margin-bottom:4px">What the output shows</div>The cut to the Monte Carlo polygon (from "Your own polygon" above) keeps about a quarter of the driving edges of the whole file (779 of 2844).</div>

<div style="border-left:4px solid #2ea043;background:rgba(46,160,67,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#2ea043;margin-bottom:4px">Tip</div>If the download of an extract fails on your machine (some cloud networks block Geofabrik), get the file another way and use <code>pbf=</code>.</div>

## The build options

These keywords work in `duckosm.network`, `area.build` and `duckosm.build`. Each one sets one key of the config file.

| Keyword | Default | What it does |
|---|---|---|
| `speeds` | `True` | `False`: no `maxspeed_kmh` (the raw OSM `maxspeed` text stays), and no `cost_s` |
| `costs` | `True` | `False`: no `cost_s`, the travel time of each edge |
| `restrictions` | `True` | `False`: no turn restrictions (`driving.turn_restrictions`) |
| `bus_routes` | `True` | `False`: no bus routes (`bus.routes`, `bus.route_edges`) |
| `connect_paths` | `True` | `False`: the free ends of paths stay free; `True` connects each one to the nearest node |
| `snap_m` | `10` | the largest distance, in metres, for `connect_paths` |
| `features` | `True` | `False`: no base-map layers (`features.*`: water, land, buildings) |
| `levels` | `True` | `False`: no drawing order of the roads (bridges over tunnels) |
| `dem` | none | a DEM file (GeoTIFF): the heights of the nodes and of the edges, after the build |

A smaller build is faster. Here the build has no bus routes, no base map and no drawing order.

In [14]:
small = duckosm.network(osm_id=2220322, name="monaco_small", modes=["driving"],
                        bus_routes=False, features=False, levels=False)       # three options off
sorted({t.split(".")[0] for t in small.tables()})                           # the schemas of the file

cut: 34461 nodes, 5685 ways, 339 relations

Output: data/monaco_small.duckdb

['driving', 'main']

<div style="border-left:4px solid #388bfd;background:rgba(56,139,253,0.10);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#388bfd;margin-bottom:4px">What the output shows</div>The file has only <code>driving</code> and <code>main</code>: no <code>bus</code> (no bus routes), no <code>features</code> and <code>raw</code> (no base map) and no <code>visualization</code> (no drawing order). Compare it with the schemas of <code>net</code> from the quick start.</div>

In [15]:
sorted({t.split(".")[0] for t in net.tables()})          # the schemas of a build with the defaults

['bus', 'driving', 'features', 'main', 'raw', 'visualization', 'walking']

### Every other option: `config=`

The config file has many more keys (`config/template.yaml` in the repository lists them all). `config=` takes such a
file (YAML or JSON) or a Python dict. A keyword wins over the same key of the config.

<div style="border-left:4px solid #d29922;background:rgba(210,153,34,0.14);border-radius:6px;padding:10px 14px;margin:10px 0;font-size:0.95em;line-height:1.55"><div style="font-weight:700;color:#d29922;margin-bottom:4px">Warning</div>A wrong keyword stops the call with the nearest name, for example <code>network has no argument 'bus_route': did you mean 'bus_routes'?</code>.</div>

In [16]:
cfg = {"options": {"bus_routes": False}, "clip": {"connect_snap_m": 5}}      # two keys of the config file
tuned = duckosm.network(osm_id=2220322, name="monaco_tuned", modes=["walking"], config=cfg,
                        features=False, levels=False)
tuned.summary()

cut: 34461 nodes, 5685 ways, 339 relations

Output: data/monaco_tuned.duckdb

,mode,edges,nodes,km,oneway_km
0,walking,11594,4587,258.0,0.0


In [17]:
try:
    duckosm.network(osm_id=2220322, name="x", modes=["driving"], bus_route=False)    # a wrong name
except duckosm.DuckOSMError as e:
    print(e)

network has no argument 'bus_route': did you mean 'bus_routes'? The area: osm_id, bbox, h3_cell, boundary, point, dist_m, cache, refresh. The build options: speeds, costs, restrictions, bus_routes, connect_paths, snap_m, config, dem, features, levels.


## Open the file again

The file stays on disk. `duckosm.open` gives its `Network` in a new session, without a build.

In [18]:
again = duckosm.open("data/monaco.duckdb")      # the file of the quick start
again.summary()

,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5
1,walking,11938,4589,260.6,0.0


## Where the files go

| File | Where | Set it with |
|---|---|---|
| the `.duckdb` file | `<out>/<name>.duckdb` (default `data/<name>.duckdb`) | `out=`, `name=`, or `output=` in `build` |
| the cut OSM file and the boundary | `<out>/<name>.osm.pbf`, `<out>/<name>.geojson` | `out=`, `name=` |
| the Geofabrik extracts | the cache: `~/.cache/duckosm` | the setting `DUCKOSM_CACHE` |

## Try it yourself

1. Get the walking network of 1 km around a place that you know, with `point=` and `dist_m=1000`.
2. Build Monaco twice, once with `speeds=False`, and compare `net.columns("driving.edges")` of both files.
3. Use `dry_run=True` for your own city, and look at the size of its extract before you get it.

## Summary

| Call | Does |
|---|---|
| `duckosm.network(...)` | gets and builds an area in one call; gives a `Network` |
| `duckosm.get(...)` | the boundary and the cut OSM file; gives an `Area` (`dry_run=True`: only a plan) |
| `area.build(modes=...)` | builds an `Area` |
| `duckosm.build(pbf=..., boundary=...)` | builds from files |
| `duckosm.AreaPicker()` | chooses an area on a map |
| `duckosm.open(path)` | opens a file that you built before |
| `net.summary()`, `net.modes`, `net.walking` | the modes of a file |

The reference: [Python SDK, Core](../reference/python.md). **Next:** [2. Look inside the file](02_inspect.md), the
tables, the columns and SQL.